# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [18]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [19]:
# Quiet ensures your UI is not full of logs.
!pip install -q "google-genai==2.12.1" "google-auth==2.49.0"
print("Done.")

Done.


In [20]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "idli_recipe": """
    Idli is a traditional South Indian breakfast dish made using
    fermented rice and urad dal batter.
    The rice and urad dal are soaked separately for several hours.
    They are ground into a smooth batter and mixed together.
    The batter is fermented overnight in a warm place.
    The fermented batter is poured into idli moulds and steamed
    until the idlis become soft and fluffy.
    """,

    "masala_dosa": """
    Masala dosa is a South Indian dish made using fermented
    rice and urad dal batter.
    The batter is spread thinly over a hot greased tawa.
    A potato filling is prepared using boiled potatoes, onions,
    mustard seeds, curry leaves and turmeric.
    The dosa is cooked until crispy and golden brown.
    The potato filling is placed inside the dosa before serving.
    """,

    "paneer_butter_masala": """
    Paneer butter masala is a popular North Indian curry.
    It is prepared using paneer cubes, tomatoes, onions,
    butter, cream and aromatic spices.
    Tomatoes and onions are cooked and blended into a smooth gravy.
    Butter, spices and cream are added to the gravy.
    Paneer cubes are added and simmered until they absorb the flavours.
    """,

    "vegetable_pulao": """
    Vegetable pulao is a rice dish cooked with vegetables and spices.
    Basmati rice is washed and soaked before cooking.
    Onions and whole spices are sautéed in oil or ghee.
    Chopped vegetables such as carrots, peas and beans are added.
    Rice and water are added and cooked until the rice becomes fluffy.
    The pulao is served hot with raita or curry.
    """
}

print("Number of documents:", len(DOCUMENTS))

Number of documents: 4


In [21]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for relevant information."""

    query_words = set(query.lower().split())
    results = []

    for doc_name, content in DOCUMENTS.items():
        content_lower = content.lower()

        match_count = sum(
            1 for word in query_words
            if len(word) > 2 and word in content_lower
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found."

    output = []

    for score, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Test the search tool
print(search_documents("How is dosa batter prepared?"))

[Source: masala_dosa]

    Masala dosa is a South Indian dish made using fermented
    rice and urad dal batter.
    The batter is spread thinly over a hot greased tawa.
    A potato filling is prepared using boiled potatoes, onions,
    mustard seeds, curry leaves and turmeric.
    The dosa is cooked until crispy and golden brown.
    The potato filling is placed inside the dosa before serving.
    

[Source: idli_recipe]

    Idli is a traditional South Indian breakfast dish made using
    fermented rice and urad dal batter.
    The rice and urad dal are soaked separately for several hours.
    They are ground into a smooth batter and mixed together.
    The batter is fermented overnight in a warm place.
    The fermented batter is poured into idli moulds and steamed
    until the idlis become soft and fluffy.
    


In [22]:
# @title Ask gemini to search for something
from google import genai
from google.genai import types

MODEL_NAME = "gemini-flash-lite-latest"
client = genai.Client(api_key=api_key)

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the recipe documents for relevant information.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The information to search for."
                    }
                },
                "required": ["query"]
            }
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
    system_instruction="""
    You are a helpful chef assistant.

    Answer questions about recipes using the search_documents tool.

    When a question requires recipe-specific information,
    call the search_documents tool.

    Only use information found in the documents.

    If the documents do not contain the answer,
    clearly state that the information is unavailable.

    Do not invent recipe information.
    """
)

print("Gemini model ready:", MODEL_NAME)

Gemini model ready: gemini-flash-lite-latest


In [23]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError("API key not found. Check Colab Secrets.")

os.environ["GOOGLE_API_KEY"] = api_key

print("API key loaded successfully!")

API key loaded successfully!


In [27]:
run_agent("How is idli batter prepared and fermented?")

Question: How is idli batter prepared and fermented?
--------------------------------------------------

Step 1: Calling search_documents
Search query: {'query': 'idli batter preparation fermentation'}
Search result:
 [Source: idli_recipe]

    Idli is a traditional South Indian breakfast dish made using
    fermented rice and urad dal batter.
    The rice and urad dal are soaked separately for several hours.
    They are ground into a smooth batter and mixed together.
    The batter is fermented overnight in a w

Final Answer:
 To prepare and ferment idli batter:

1. **Soak:** Rice and urad dal are soaked separately for several hours.
2. **Grind & Mix:** They are ground into a smooth batter and then mixed together.
3. **Ferment:** The batter is left to ferment overnight in a warm place. 

After fermentation, the batter is ready to be poured into idli moulds and steamed.


'To prepare and ferment idli batter:\n\n1. **Soak:** Rice and urad dal are soaked separately for several hours.\n2. **Grind & Mix:** They are ground into a smooth batter and then mixed together.\n3. **Ferment:** The batter is left to ferment overnight in a warm place. \n\nAfter fermentation, the batter is ready to be poured into idli moulds and steamed.'

In [28]:
questions = [
    "How is idli batter prepared and fermented?",
    "What ingredients are used in paneer butter masala?",
    "How do you prepare vegetable pulao?",
    "How many ghosts eat carrots everyday?",
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):

    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)
    run_agent(q)


Question: How is idli batter prepared and fermented?
--------------------------------------------------

Step 1: Calling search_documents
Search query: {'query': 'idli batter preparation fermentation'}
Search result:
 [Source: idli_recipe]

    Idli is a traditional South Indian breakfast dish made using
    fermented rice and urad dal batter.
    The rice and urad dal are soaked separately for several hours.
    They are ground into a smooth batter and mixed together.
    The batter is fermented overnight in a w

Final Answer:
 To prepare and ferment idli batter:

1. **Soaking:** Rice and urad dal are soaked separately for several hours.
2. **Grinding & Mixing:** They are ground into a smooth batter and then mixed together.
3. **Fermentation:** The batter is left to ferment overnight in a warm place. 

Once fermented, the batter is ready to be poured into idli moulds and steamed.

Question: What ingredients are used in paneer butter masala?
-------------------------------------------

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [29]:
run_agent("What is the most popular food item in Canada?")

Question: What is the most popular food item in Canada?
--------------------------------------------------

Step 1: Calling search_documents
Search query: {'query': 'most popular food item in Canada'}
Search result:
 [Source: paneer_butter_masala]

    Paneer butter masala is a popular North Indian curry.
    It is prepared using paneer cubes, tomatoes, onions,
    butter, cream and aromatic spices.
    Tomatoes and onions are cooked and blended into a smooth gravy.
    Butter, spices and cream are added to the 

Final Answer:
 The provided documents do not contain information about the most popular food item in Canada.


'The provided documents do not contain information about the most popular food item in Canada.'

In [26]:
import time

def run_agent(question: str, max_steps: int = 8) -> str:

    print("Question:", question)
    print("-" * 50)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=question)]
        )
    ]

    for step in range(1, max_steps + 1):

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=contents,
            config=config
        )

        model_content = response.candidates[0].content
        contents.append(model_content)

        calls = [
            part.function_call
            for part in model_content.parts
            if part.function_call
        ]

        if not calls:
            final_answer = response.text or "No final answer generated."
            print("\nFinal Answer:\n", final_answer)
            return final_answer

        result_parts = []

        for call in calls:

            args = dict(call.args)

            print(f"\nStep {step}: Calling {call.name}")
            print("Search query:", args)

            if call.name == "search_documents":
                result = search_documents(**args)
            else:
                result = "Unknown tool requested."

            print("Search result:\n", result[:300])

            result_parts.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        contents.append(
            types.Content(
                role="user",
                parts=result_parts
            )
        )

    return "Agent stopped: maximum steps reached."

In [ ]:
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key not found. Check the secret name.")

API key loaded successfully!


In [ ]:
run_agent("How is idli batter prepared and fermented?")

Question: How is idli batter prepared and fermented?
--------------------------------------------------


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)